# Select Nouns from COCO corpus

In [ ]:
import pandas as pd
# df = pd.read_csv("coco_text_image_clip_embeddings.csv") - data missing due to size limit


In [11]:
import pandas as pd
import spacy
from collections import Counter
from nltk.corpus import wordnet as wn


nlp = spacy.load("en_core_web_sm")

def get_strict_compounds(df, column="caption"):
    all_terms = []
    
    print(f"Processing {len(df)} rows... Filtering for strict noun compounds.")
    
    # Disable NER for speed
    for doc in nlp.pipe(df[column].astype(str), batch_size=500, disable=["ner"]):
        
        # 1. Capture Single Nouns (if you still want them)
        single_nouns = [t.lemma_.lower() for t in doc if t.pos_ == "NOUN" and not t.is_stop]
        all_terms.extend(single_nouns)

        # 2. Capture ONLY 2-word Noun-Noun Compounds
        for token in doc:
            # Look for a word that is modifying another word as a 'compound'
            # Example: 'roller' (compound) -> 'coaster' (head)
            if token.dep_ == "compound" and token.head.pos_ in ["NOUN", "PROPN"]:
                
                # Ensure the modifier itself is also a NOUN/PROPN (blocks adjectives/verbs)
                if token.pos_ in ["NOUN", "PROPN"]:
                    
                    # Verify it's a 2-word sequence (optional but keeps it to your '2 word' rule)
                    # This creates the phrase: [modifier] [head]
                    phrase = f"{token.text.lower()} {token.head.text.lower()}"
                    
                    # One last check: no stop words (blocks "the house" etc)
                    if not token.is_stop and not token.head.is_stop:
                        all_terms.append(phrase)
                        #print(phrase)

    counts = Counter(all_terms)
    counts_df = pd.DataFrame(counts.items(), columns=['Term', 'Frequency'])
    return counts_df.sort_values(by='Frequency', ascending=False).reset_index(drop=True)

result = get_strict_compounds(df, "caption")

Processing 117266 rows... Filtering for strict noun compounds.


In [14]:

def check_compound(term):
    term = str(term).lower().replace(" ", "_")
    synsets = wn.synsets(term, pos=wn.NOUN)

    if synsets:
        return True
result["decision"] = result["Term"].apply(check_compound)
result = result[result["decision"] == True]
print(len(result))

6336


In [15]:
terms = set(result["Term"].str.lower())

def is_plural_to_remove(term):
    term = term.lower()
    
    # only handle simple cases: ends with 's' but not 'ss'
    if term.endswith("s") and not term.endswith("ss"):
        singular = term[:-1]
        return singular in terms
    
    return False

# filter dataframe
result = result[~result["Term"].apply(is_plural_to_remove)]

In [16]:
result = result[result["Term"].str.len() >= 3]
print(len(result))

6002


# Filter by selected Nouns

In [17]:
import re
from tqdm import tqdm

df = df.copy()
df["caption_lower"] = df["caption"].str.lower()

used_captions = set()
filtered_list = []

for concept in tqdm(result["Term"].dropna(), desc="Processing concepts"):
    concept_lower = concept.lower()
    concept_pattern = rf"\b{re.escape(concept_lower)}\b"

    # vectorized mask (fast)
    mask = df["caption_lower"].str.contains(concept_pattern, na=False, regex=True)
    matches = df[mask].copy()

    if matches.empty:
        continue

    # compute position (vectorized)
    matches["match_pos"] = matches["caption_lower"].str.find(concept_lower)

    # sort by earliest appearance
    matches = matches.sort_values("match_pos")

    # pick first unused caption (still a loop, but smaller now)
    for _, row in matches.iterrows():
        caption = row["caption"]
        if caption not in used_captions:
            filtered_list.append(row.drop(labels="match_pos").to_frame().T)
            used_captions.add(caption)
            break

filtered_by_concept = (
    pd.concat(filtered_list, ignore_index=True)
    if filtered_list else pd.DataFrame(columns=df.columns)
)

Processing concepts: 100%|██████████| 6002/6002 [19:38<00:00,  5.09it/s]


In [18]:
filtered_by_concept = filtered_by_concept.drop_duplicates(subset=["caption"], keep="last")
len(filtered_by_concept)

5413

In [ ]:
filtered_by_concept = filtered_by_concept.drop_duplicates(subset=["caption"], keep="last")
len(filtered_by_concept)

5390

In [10]:
for idx, row in filtered_by_concept.iterrows():
    if "Monkey" in row["caption"]: print(idx, row["caption"])

894 Monkey on a string climbing towards a red and green vase.
4198 A baseball trading car depicting a pitcher for the Steam Monkeys minor league baseball team.


In [11]:
results = []

for concept in result["Term"]:
    count = 0
    for cap in filtered_by_concept["caption"]:
        if concept.lower() in cap.lower():
            count += 1
    results.append((concept, count))

# sort by count (descending)
results.sort(key=lambda x: x[1], reverse=True)

# print
for concept, count in results:
    if len(concept.split()) < 2: continue
    #if count < 20: continue
    print(concept, count)

group o 56
teddy bear 47
living room 40
hot dog 32
fire hydrant 31
clock tower 26
baseball play 25
baseball player 24
street sign 23
traffic light 22
parking lot 22
ball game 22
young man 19
parking meter 18
baseball game 17
tennis player 16
park bench 13
laptop computer 12
ice cream 12
video game 11
ball field 11
cutting board 9
remote control 9
tennis ball 8
tennis racket 8
baseball bat 8
baseball field 8
birthday cake 8
tennis match 8
train station 7
soccer ball 7
bus stop 7
computer monitor 7
coffee table 7
christmas tree 7
paper plate 7
toilet bowl 7
french fries 7
ski slope 6
street corner 6
toilet seat 6
palm tree 6
paper towel 6
american flag 6
red light 6
red fire 6
white horse 6
bird feed 6
tennis court 5
home plate 5
computer keyboard 5
hotel room 5
dining room 5
display case 5
wedding cake 5
school bus 5
fire truck 5
cross country 5
shower curtain 5
dirt bike 5
jet plane 5
bird feeder 5
microwave oven 5
toaster oven 5
desktop computer 4
tree branch 4
big ben 4
toilet paper 

In [ ]:
import numpy as np

# ---------- Seed ----------
np.random.seed(42)

# ---------- Parse ----------
def parse(col):
    return np.vstack(
        col.apply(
            lambda x: np.fromstring(
                x.replace("\n", " ").strip("[]"),
                sep=" ",
                dtype=np.float32
            )
        ).values
    )

X_text = parse(filtered_by_concept["clip_text_embedding"])
X_image = parse(filtered_by_concept["clip_embedding"])

# ---------- Normalize (CRITICAL for CLIP) ----------
def normalize(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1e-8
    return X / norms

X_text = normalize(X_text)
X_image = normalize(X_image)
# print first 10 values of text embedding for sanity check

# ---------- FUSE (EARLY FUSION like paper) ----------
# Option 1: simple concat
X_fused = np.hstack([X_text, X_image])

# Optional: weighting (uncomment if needed)
# alpha = 0.5
# X_fused = np.hstack([alpha * X_image, (1 - alpha) * X_text])

# Normalize again after fusion (important)
X_fused = normalize(X_fused)

# ---------- Sampling (COSINE DISTANCE) ----------
n_samples = len(filtered_by_concept)
N = X_fused.shape[0]

selected_indices = []

# Start random
first = np.random.randint(0, N)
selected_indices.append(first)

# Cosine distance = 1 - cosine similarity
distances = 1 - (X_fused @ X_fused[first])

for i in range(1, n_samples):
    if i % 50 == 0:
        print(f"Selected {i}/{n_samples}")

    # pick farthest point (max cosine distance)
    next_idx = np.argmax(distances)
    selected_indices.append(next_idx)

    # compute cosine distance to new point
    new_dist = 1 - (X_fused @ X_fused[next_idx])

    # update minimum distances
    distances = np.minimum(distances, new_dist)

# ---------- Final ----------
df_diverse = filtered_by_concept.iloc[selected_indices].reset_index(drop=True)

In [33]:
df_filtered = df_diverse[:500]